# 02 — Assertions and Golden Outputs

## Why this notebook exists

In **`01_why_agents_are_hard_to_test.ipynb`** we saw how a naive `assert agent(x) == "expected"` unit test flakes the moment an agent varies its phrasing — and we named the three forces behind that fragility: non-determinism, no single ground truth, and multi-step compounding failures. The closing insight was: *"We can't `assert equals` our way out of this. We need graders that score outputs, not match them exactly."*

This notebook builds those graders — the cheapest, most reliable layer of the eval stack. Each grader is a plain callable that accepts an example and a candidate output and returns a `Score`. No LLM, no external service, no API key required.

## What you'll learn

- The `Score` dataclass — the shared return type every grader in this series produces.
- When **exact match** is appropriate (structured outputs, deterministic pipelines) and why it's brittle for free-text.
- How **contains** and **regex** graders give you partial-match power without LLM overhead.
- How to validate **structured outputs** with `pydantic` v2 — confirming an agent's JSON parses into the expected schema, and capturing the validation error when it doesn't.
- The discipline of **golden outputs**: storing a known-good string to a file and diffing against it — and why keeping goldens from silently going stale is as important as writing them.
- How all four grader families share a single `(example, output) -> Score` signature, so they compose transparently in notebook 03's harness.

## 1. Setup

All graders in this notebook share one return type: `Score`. Defining it here, once, as a `dataclass` gives us a consistent interface that notebook 03 will rely on when it builds the harness.

**Compatibility note:** In this notebook examples are plain dicts — `{"input": ..., "expected": ...}` — and graders read `example["expected"]`. Notebook 03 will formalize `Example` as a proper dataclass with `.input` and `.expected` attributes, at which point `example["expected"]` becomes `example.expected`. The grader signatures don't change — only how you access the field. This is called out again at each grader definition below.

If `pydantic` isn't installed yet:
```bash
pip install pydantic
```

In [ ]:
import re
import tempfile
from dataclasses import dataclass
from pathlib import Path

from pydantic import BaseModel, ValidationError
from typing import Literal

# Track every temp file written in this notebook for cleanup at the end.
_temp_files: list[Path] = []


@dataclass
class Score:
    key: str            # grader identifier, e.g. "exact_match"
    score: float        # normalized to [0.0, 1.0]
    passed: bool
    comment: str = ""


print("Setup OK")
print(f"Score fields: {[f.name for f in Score.__dataclass_fields__.values()]}")